# Deploy a single model on SageMaker Endpoint with LMI container using AWS Python API (boto3)


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import json
import time
import boto3
from botocore.loaders import Loader

REGION = "us-west-2"
MODEL_S3_URI = "s3://sagemaker-benchmark-lumen-model-artifact/gpt-oss-20b/"
ROLE_ARN = "arn:aws:iam::XXXXXXXXXXXX:role/SageMaker-Benchmark-Lumen"
S3_OUTPUT = "s3://sagemaker-benchmark-lumen-output/recommendation-output/"
CONFIG_NAME = "ds-rec-config-11"
JOB_NAME = "ds-rec-job-11"

# Load the custom service model into botocore
loader = Loader()
api = json.load(open("./sagemaker-2017-07-24.normal.json"))
session = boto3.Session(region_name=REGION)

# Create client with custom service model
sm = session.client("sagemaker",
                    endpoint_url="https://api.sagemaker.us-west-2.amazonaws.com",
                    api_version=api.get("metadata", {}).get("apiVersion"))

## Lumen setup

In [ ]:
# 1. Create workload config
workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "custom_dataset_type": "generic",
        "input_file": "/opt/ml/input/data/datasets/gsm8k_training.jsonl",
        "concurrency": 64,
        "benchmark_duration": 300,
        "streaming": True,
        "warmup_duration": 30,
        "output_tokens_mean": 200,
        "output_tokens_stddev": 0,
        "extra_inputs": "ignore_eos:true temperature:0",
        "prompt_input_tokens_mean": 3500,
        "random_seed": 42, 
    },
}

config_resp = sm.create_ai_workload_config(
    AIWorkloadConfigName=CONFIG_NAME,
    DatasetConfig={
        "InputDataConfig": [
            {
                "ChannelName": "datasets",
                "DataSource": {
                    "S3DataSource": {
                        # This should be folder path, not file path
                        "S3Uri": "s3://sagemaker-benchmark-lumen-dataset/dataset/"
                    }
                }
            }
        ]
    },
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)
print(f"Config created: {config_resp['AIWorkloadConfigArn']}")

In [ ]:
# 2. Create recommendation job
job_resp = sm.create_ai_recommendation_job(
    AIRecommendationJobName=JOB_NAME,
    ModelSource={
        "S3": {
            "S3Uri": MODEL_S3_URI
            
        }
    },
    OutputConfig={"S3OutputLocation": S3_OUTPUT},
    AIWorkloadConfigIdentifier=CONFIG_NAME,
    RoleArn=ROLE_ARN,
    PerformanceTarget={
        "Constraints": [
            {"Metric": "throughput"},
        ]
    },
    
    ComputeSpec={
        "InstanceTypes": ["ml.p5en.48xlarge"],
        "CapacityReservationConfig": {
            "CapacityReservationPreference": "capacity-reservations-only",
            "MlReservationArns": [
                "arn:aws:sagemaker:us-west-2:276831242074:training-plan/lumen-test-1",
                "arn:aws:sagemaker:us-west-2:276831242074:training-plan/lumen-test"
                
            ]
        }
    },

    OptimizeModel=True,
    InferenceSpecification = {"Framework": "LMI"},
)
print(f"Job created: {job_resp['AIRecommendationJobArn']}")

In [ ]:
# 3. Poll until terminal
while True:
    desc = sm.describe_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)
    status = desc["AIRecommendationJobStatus"]
    print(f"  Status: {status}")
    if status in ("Completed", "Failed", "Stopped"):
        print(desc)
        break
    time.sleep(30)

In [ ]:
# 4. Print recommendations
if status == "Completed":
    recommendations = desc.get("Recommendations", [])
    print(f"\n{len(recommendations)} recommendation(s) found:")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n--- Recommendation {i} ---")
        print(f"  {rec.get('RecommendationDescription', 'N/A')}")
        if rec.get("InstanceDetails"):
            for inst in rec["InstanceDetails"]:
                print(f"  Instance: {inst['InstanceType']} x{inst.get('InstanceCount', 1)}")
        if rec.get("OptimizationDetails"):
            for opt in rec["OptimizationDetails"]:
                print(f"  Optimization: {opt['OptimizationType']}")
        if rec.get("ExpectedPerformance"):
            for perf in rec["ExpectedPerformance"]:
                print(f"  {perf['Metric']}: {perf['Value']} {perf.get('Unit', '')}")
        if rec.get("DeploymentConfiguration"):
            deploy = rec["DeploymentConfiguration"]
            print(f"  Deploy image: {deploy.get('ImageUri', 'N/A')}")
            print(f"  Deploy instance: {deploy.get('InstanceType', 'N/A')}")
else:
    print(f"\nEnded with status: {status}")
    if "FailureReason" in desc:
        print(f"Reason: {desc['FailureReason']}")

In [ ]:
print(json.dumps(desc.get("Recommendations", []), indent=2))

In [ ]:
sm.stop_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
#
# Cleanup
#
sm.delete_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
sm.delete_ai_workload_config(AIWorkloadConfigName=CONFIG_NAME)